# Exercise 3: Change a FATES complexity variable

FATES can run in several "complexity modes" that change how plant functional types (PFTs) are assigned to patches. Three common ones are controlled by namelist variables in ``user_nl_clm``:

- ``use_fates_sp``: turns on satellite phenology mode
- ``use_fates_nocomp``: turns on "no competition" mode, where each patch holds a single PFT and PFT area is prescribed rather than emerging from competition.
- ``use_fates_fixed_biogeog``: turns on fixed biogeography, where the PFTs allowed at each grid cell are read from the surface dataset rather than determined by the model.

These are logical variables, so they take ``.true.`` or ``.false.``.


<div class="alert alert-info">

## Exercise: Choose a FATES complexity mode
 
Create a new case named `i2000_fates_nocomp` using the compset  `I2000Clm60FatesCrujraRsGs` and resolution `f19_f19_mt233`.

Turn on no competition and fixed biogeography.

Run ``./preview_namelists`` and check that ``CaseDocs/lnd_in`` shows both variables set to ``.true.``. Then run the case for 1 month.

</div>


<div class="alert alert-warning">  
<details>
   <summary>
      <font face="Times New Roman" color='blue'>Click here for hints</font>
    </summary>


**Where are the parameters?**

The variable you are looking for is:

```fortran
use_fates_nocomp
use_fates_fixed_biogeog
```

It can be modified directly in `user_nl_clm`.

**Compare this run with the FATES SP run**

Compare this run with the case you created in the previous exercise:

```text
i2000_fates_sp.basics
```

Both cases produce output on a regular **2° × 2° latitude–longitude grid**, making them easy to compare using NetCDF tools.

You can use **ncdiff** to compute differences:

```bash
ncdiff /glade/derecho/scratch/$USER/archive/i2000_fates_sp.basics/lnd/hist/i2000_fates_sp.basics.cam.h0a.0001-01.nc \
/glade/derecho/scratch/$USER/archive/i2000_fates_nocomp/lnd/hist/b1850_interp.cam.h0a.0001-01.nc \
diff.nc
```

Then visualize the result:

```bash
ncview diff.nc
```

How does changing the FATES complexity mode affect the **FATES_LAI** field?
</details>
</div>



<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

**Create new case**

```bash
cd /glade/u/home/$USER/code/my_clm_code/cime/scripts/

./create_newcase \
  --case ~/cases/i2000_fates_nocomp \
  --compset I2000Clm60FatesCrujraRsGs \
  --res f19_f19_mt233 \
  --run-unsupported
```

**Setup**

```bash
cd ~/cases/i2000_fates_nocomp
./case.setup
```

**Customize the CLM namelist**

Edit `user_nl_clm` and add:

```fortran
use_fates_nocomp = .true.
use_fates_fixed_biogeog = .true.
```

**Set the run length**

```bash
./xmlchange STOP_N=1
./xmlchange STOP_OPTION=nmonths
```

**Change the wallclock time, job queue and project number**

If needed:

```bash
./xmlchange JOB_WALLCLOCK_TIME=01:00:00
./xmlchange JOB_QUEUE=tutorial
./xmlchange PROJECT=UESM0016 --force
```

Use the project number provided for the tutorial.

**Build and submit**

```bash
qcmd -- ./case.build
./case.submit
```

**Analyze the results**

When the run completes, compare it with the `b1850_interp` case.

Create a file containing the differences:

```bash
cd /glade/derecho/scratch/$USER/archive/b1850_dcs

ncdiff \
/glade/derecho/scratch/$USER/archive/b1850_dcs/atm/hist/b1850_dcs.cam.h0a.0001-01.nc \
/glade/derecho/scratch/$USER/archive/b1850_interp/atm/hist/b1850_interp.cam.h0a.0001-01.nc \
diff.nc
```

Visualize the differences:

```bash
 module load ncview
 ncview diff.nc
```

Display the **LWCF** field and examine how it changes.

**# Discussion**

**Dcs** is the threshold diameter at which cloud ice particles are converted to snow.

Increasing

```fortran
micro_mg_dcs = 450.D-6
```

to

```fortran
micro_mg_dcs = 800.D-6
```

allows cloud ice particles to grow larger before being converted to snow. As a result, more cloud ice tends to remain suspended in the atmosphere, which generally increases the **Longwave Cloud Forcing (LWCF)**.

The largest changes are typically found in the tropical deep convective regions, where high ice clouds are abundant.

Keep in mind that this simulation is only one month long. Although you should see differences in the LWCF field, a one-month simulation is too short to draw statistically robust conclusions about the climate response.


**# How does this affect the LWCF?** 

Dcs = Threshold diameter to convert cloud ice particles to snow

*<p style="text-align: center;"> 
    ![dcs representation](../../../../images/namelist/dcs.png)  
    Figure: Representation of Dcs. </p>*
    
We increased Dcs from 450 microns to 800 microns. We should have more ice cloud and the LWCF should be larger. 

  
The field `LWCF` difference looks like 
*<p style="text-align: center;"> 
    ![ncview diff LWCF](../../../../images/namelist/ncview_diff_dcs.png)  
    Figure: Difference of LWCF for micro_mg_dcs = 450.D-6->800.D-6 </p>*
    
We can see that the change in Dcs affects the tropics where LWCF is large. However, a 1-month run is too short to look at robust statistics. 

</details>
</div>

